In [0]:
from pyspark.sql.functions import col, date_format, lit,year,month,dayofmonth,hour,row_number, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime
import uuid

In [0]:
input_table_path = dbutils.widgets.get("input_table_path")
id_col = dbutils.widgets.get("ID_COL")
ground_truth_col = dbutils.widgets.get("GROUND_TRUTH_COL")
ground_truth_table = dbutils.widgets.get("GROUND_TRUTH_TABLE")
job_frequency = dbutils.widgets.get("JOB_FREQUENCY")
project_name = dbutils.widgets.get("project_name")

In [0]:
uuid = uuid.uuid4().hex

In [0]:
def merge_with_delta_table(table_name, new_df, job_frequency, id_column="id"):
    """ Merge new records into Delta table with duplicate prevention """
    from delta.tables import DeltaTable

    # Add time partition to new data
    if job_frequency == 'hourly':
        new_df = new_df.withColumn("time_partition", date_format(col("timestamp"), "yyyy-MM-dd-HH"))
    elif job_frequency == 'daily':
        new_df = new_df.withColumn("time_partition", date_format(col("timestamp"), "yyyy-MM-dd"))
    elif job_frequency == 'monthly':
        new_df = new_df.withColumn("time_partition", date_format(col("timestamp"), "yyyy-MM"))
    elif job_frequency == 'yearly':
        new_df = new_df.withColumn("time_partition", date_format(col("timestamp"), "yyyy"))

    # Read existing Delta table
    delta_table = DeltaTable.forName(spark, table_name)

    # Add time partition to existing table
    existing_with_partition = delta_table.toDF()
    if job_frequency == 'hourly':
        existing_with_partition = existing_with_partition.withColumn(
            "time_partition", date_format(col("timestamp"), "yyyy-MM-dd-HH"))
    elif job_frequency == 'daily':
        existing_with_partition = existing_with_partition.withColumn(
            "time_partition", date_format(col("timestamp"), "yyyy-MM-dd"))
    elif job_frequency == 'monthly':
        existing_with_partition = existing_with_partition.withColumn(
            "time_partition", date_format(col("timestamp"), "yyyy-MM"))
    elif job_frequency == 'yearly':
        existing_with_partition = existing_with_partition.withColumn(
            "time_partition", date_format(col("timestamp"), "yyyy"))

    # Anti-join to find records that don't exist in the same time partition
    records_to_insert = new_df.alias("new").join(
        existing_with_partition.alias("existing"),
        (col(f"new.{id_column}") == col(f"existing.{id_column}")) &
        (col("new.time_partition") == col("existing.time_partition")),
        "left_anti"
    ).drop("time_partition")

    # Append only new records
    # records_to_insert.write.format("delta").mode("append").saveAsTable(table_name)
    return records_to_insert


In [0]:
df = raw_data.withColumn("uuid", lit(uuid)) \
  .withColumn("id",col(id_col).cast("string")) \
  .withColumnRenamed(ground_truth_col,"ground_truth") \
  .withColumn("project_name",lit(project_name)) \
  .withColumn("timestamp", current_timestamp()) \
  .select("uuid","id","ground_truth","project_name","timestamp")

if spark.catalog.tableExists(ground_truth_table):
  result = merge_with_delta_table(table_name, df, job_frequency, "id")
else:
  df.write.mode("overwrite").saveAsTable(ground_truth_table)